# Prefix Alarm Monitor Benchmark

This notebook is a compact entry point for the systematic scorer, feature-ablation, and alarm-rule comparison. Implementations live under `prefix_alarm_monitor/`; the notebook only configures, runs, and displays the benchmark.

All thresholds are selected on pure validation leaf trajectories. Test metrics are reported without retuning.

## Compared Monitors

Each scorer is evaluated with both `full_features` and `no_confidence`:

- Linear: calibrated, tuned CUSUM, gated/LCB, consecutive-$K$, EWMA, leaky CUSUM, and sliding-window risk (14 rows).
- LightGBM: calibrated, tuned CUSUM, consecutive-$K$, EWMA, leaky CUSUM, and sliding-window risk (12 rows).
- Lightweight MLP: the same six supported rules as LightGBM (12 rows).
- XGBoost: the same six supported rules as LightGBM (12 rows).

The original coarse-search CUSUM is excluded. LightGBM, MLP, and XGBoost do not expose the Linear model's confidence-bound gate, so no gated rows are shown for those scorers.

## 1. Configure

The existing data protocol is unchanged: mixed-root files train the prefix scorer, while stratified pure-root files are used for validation and test. Prefixes use a trailing 64-step window, stride 8, and skip steps 1 through 8.

The four result tables report test prefix-level AUROC for each scorer. AUROC is computed from continuous `p_fail = 1 - p_success`, so it is identical across alarm rules that share the same scorer and feature setup.

In [16]:
from pathlib import Path
import importlib
import json

import pandas as pd
from IPython.display import Markdown, display

import prefix_alarm_monitor.rules as rules_module
import prefix_alarm_monitor.comparison as benchmark_module

# Reload local modules so a long-lived notebook kernel cannot run stale benchmark code.
importlib.reload(rules_module)
benchmark_module = importlib.reload(benchmark_module)
default_config = benchmark_module.default_config
run_benchmark = benchmark_module.run_benchmark

NOTEBOOK_ROOT = Path.cwd()
if not (NOTEBOOK_ROOT / "data").exists():
    NOTEBOOK_ROOT = NOTEBOOK_ROOT / "alarm_monitor"

config = default_config(NOTEBOOK_ROOT)
print("data dir:", config.data_dir)
print("output root:", config.output_root)
print("target leaf recall:", config.target_leaf_recall)

data dir: /home/kevinwyk/mon_agent/alarm_monitor/data
output root: /home/kevinwyk/mon_agent/alarm_monitor/monitor_results
target leaf recall: 0.85


In [17]:
comparison_path = config.output_root / "algorithm_comparison" / "algorithm_comparison.csv"
required_columns = {"scorer", "setup", "rule", "auroc"}
expected_rows = {"linear": 14, "lightgbm": 12, "mlp": 12, "xgboost": 12}

if comparison_path.exists():
    comparison_table = pd.read_csv(comparison_path)
else:
    comparison_table = pd.DataFrame()

actual_rows = (
    comparison_table.groupby("scorer").size().to_dict()
    if "scorer" in comparison_table.columns
    else {}
)
if not required_columns.issubset(comparison_table.columns) or actual_rows != expected_rows:
    comparison_table = run_benchmark(config)

missing_columns = required_columns.difference(comparison_table.columns)
actual_rows = (
    comparison_table.groupby("scorer").size().to_dict()
    if "scorer" in comparison_table.columns
    else {}
)
if missing_columns or actual_rows != expected_rows:
    raise RuntimeError(
        "Benchmark returned an incompatible table: "
        f"missing_columns={sorted(missing_columns)}, row_counts={actual_rows}. "
        "Rerun the configuration cell, then rerun this cell."
    )

rule_labels = {
    "calibrated_leaf_low_fp": "calibrated",
    "gated_leaf_low_fp": "gated_lcb",
    "cusum_tuned_low_fp": "tuned_cusum",
    "consecutive_k_low_fp": "consecutive_k",
    "ewma_low_fp": "ewma",
    "leaky_cusum_low_fp": "leaky_cusum",
    "sliding_window_risk_low_fp": "sliding_window_risk",
}
output_columns = [
    "setup",
    "rule",
    "pure0_leaf_alarm_recall",
    "pure1_leaf_false_alarm_rate",
    "pure0_leaf_alarm_median_step",
    "pure0_successful_warning_mean_early_pct",
    "auroc",
]

for scorer_name, title in (
    ("linear", "Linear scorer (14 rows)"),
    ("lightgbm", "LightGBM scorer (12 rows)"),
    ("mlp", "Lightweight MLP scorer (12 rows)"),
    ("xgboost", "XGBoost scorer (12 rows)"),
):
    scorer_table = comparison_table.loc[
        comparison_table["scorer"] == scorer_name
    ].copy()
    scorer_table["rule"] = scorer_table["rule"].replace(rule_labels)
    scorer_table = scorer_table.rename(columns={"auroc": "scorer_prefix_auroc"})
    display(Markdown(f"### {title}"))
    display(
        scorer_table[[*output_columns[:-1], "scorer_prefix_auroc"]]
        .reset_index(drop=True)
    )

### Linear scorer (14 rows)

,setup,rule,pure0_leaf_alarm_recall,pure1_leaf_false_alarm_rate,pure0_leaf_alarm_median_step,pure0_successful_warning_mean_early_pct,scorer_prefix_auroc
0,full_features,calibrated,0.843363,0.180328,40.0,38.484326,0.890433
1,full_features,gated_lcb,0.841353,0.172563,40.0,38.048963,0.890433
2,full_features,tuned_cusum,0.817350,0.103538,49.0,21.902277,0.890433
3,full_features,consecutive_k,0.847940,0.170837,41.0,36.085672,0.890433
4,full_features,ewma,0.845484,0.171700,40.0,39.439846,0.890433
5,full_features,leaky_cusum,0.823043,0.103538,49.0,22.683495,0.890433
6,full_features,sliding_window_risk,0.844702,0.173425,41.0,39.526855,0.890433
7,no_confidence,calibrated,0.851401,0.201898,41.0,36.592119,0.877872
8,no_confidence,gated_lcb,0.843809,0.192407,41.0,35.291525,0.877872
9,no_confidence,tuned_cusum,0.822039,0.103538,49.0,20.195513,0.877872


### LightGBM scorer (12 rows)

,setup,rule,pure0_leaf_alarm_recall,pure1_leaf_false_alarm_rate,pure0_leaf_alarm_median_step,pure0_successful_warning_mean_early_pct,scorer_prefix_auroc
0,full_features,calibrated,0.876075,0.199310,40.0,40.951962,0.857941
1,full_features,tuned_cusum,0.856648,0.059534,49.0,18.786531,0.857941
2,full_features,consecutive_k,0.878196,0.253667,33.0,44.024154,0.857941
3,full_features,ewma,0.875851,0.215703,40.0,41.609694,0.857941
4,full_features,leaky_cusum,0.858100,0.062123,49.0,19.571391,0.857941
5,full_features,sliding_window_risk,0.873842,0.210526,40.0,41.413270,0.857941
6,no_confidence,calibrated,0.882997,0.211389,40.0,41.304341,0.857511
7,no_confidence,tuned_cusum,0.854750,0.059534,49.0,18.678013,0.857511
8,no_confidence,consecutive_k,0.882103,0.306299,33.0,45.296074,0.857511
9,no_confidence,ewma,0.886346,0.215703,40.0,44.331625,0.857511


### Lightweight MLP scorer (12 rows)

,setup,rule,pure0_leaf_alarm_recall,pure1_leaf_false_alarm_rate,pure0_leaf_alarm_median_step,pure0_successful_warning_mean_early_pct,scorer_prefix_auroc
0,full_features,calibrated,0.859774,0.273512,32.0,49.118437,0.851101
1,full_features,tuned_cusum,0.857207,0.151855,41.0,28.642456,0.851101
2,full_features,consecutive_k,0.867143,0.315789,25.0,48.645062,0.851101
3,full_features,ewma,0.876633,0.322692,25.0,53.509227,0.851101
4,full_features,leaky_cusum,0.843921,0.139776,42.0,27.975323,0.851101
5,full_features,sliding_window_risk,0.866250,0.320104,25.0,53.230322,0.851101
6,no_confidence,calibrated,0.861784,0.196721,40.0,40.420352,0.859932
7,no_confidence,tuned_cusum,0.845037,0.097498,48.0,24.080583,0.859932
8,no_confidence,consecutive_k,0.871274,0.275237,32.0,45.795592,0.859932
9,no_confidence,ewma,0.864798,0.194996,40.0,44.936721,0.859932


### XGBoost scorer (12 rows)

,setup,rule,pure0_leaf_alarm_recall,pure1_leaf_false_alarm_rate,pure0_leaf_alarm_median_step,pure0_successful_warning_mean_early_pct,scorer_prefix_auroc
0,full_features,calibrated,0.877303,0.191544,40.0,40.623632,0.858255
1,full_features,tuned_cusum,0.856090,0.056083,49.0,19.422326,0.858255
2,full_features,consecutive_k,0.875963,0.258844,33.0,44.919385,0.858255
3,full_features,ewma,0.882773,0.224331,40.0,44.062045,0.858255
4,full_features,leaky_cusum,0.859440,0.061260,49.0,19.703194,0.858255
5,full_features,sliding_window_risk,0.879424,0.207075,40.0,43.784311,0.858255
6,no_confidence,calibrated,0.877638,0.176877,40.0,37.995717,0.855696
7,no_confidence,tuned_cusum,0.857653,0.056946,49.0,18.551015,0.855696
8,no_confidence,consecutive_k,0.886234,0.269198,33.0,45.159787,0.855696
9,no_confidence,ewma,0.883778,0.192407,40.0,41.202627,0.855696
